## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | SE-ResNeXt-50 3-stage training on original published crops |
| Model | SE-ResNeXt-50 (ImageNet pretrained, linear head) |
| Input | Original published 224x224 crops -> SquarePad -> Resize(384) |
| Training | 3-stage: frozen(5) -> coarse-tune(15) -> fine-tune(10) |
| Loss | Plain CrossEntropyLoss |
| Selection | QWK only |
| Outputs | best_model.pth, last_model.pth, history.csv, metadata.json |
| Status | Clean 3-stage training (matches working baseline pattern) |


## Detailed config

### Identity

| Item | Value |
| --- | --- |
| Purpose | Train SE-ResNeXt-50 on original published crops, 3-stage |
| Workflow | Stage 1 (frozen head) -> Stage 2 (coarse-tune) -> Stage 3 (fine-tune) |
| Output dir | `/content/drive/MyDrive/Models/seresnext50_32x4d_original/<TIMESTAMP>/` |

### Dataset

| Item | Value |
| --- | --- |
| Classes | 5 KL grades (0-4) |
| Dataset root | `/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224` |
| Input size | 384x384 |
| Augmentation | CLAHE -> SquarePad -> PIL -> HFlip(p=0.5) -> Rotation(5) -> ColorJitter(0.08,0.08) -> Resize(384) -> RandomErasing(0.10) -> ImageNet norm |

### Training

| Stage | Epochs | Head LR | Backbone LR | Scope |
| --- | --- | --- | --- | --- |
| Stage 1 | 5 | 3e-4 | 0 | head only, backbone frozen |
| Stage 2 | 15 | 3e-4 | 3e-5 | head + last conv block |
| Stage 3 | 10 | 1e-5 | 1e-5 | full fine-tune |
| Total | 30 | | | |

| Item | Value |
| --- | --- |
| Batch size | 48 |
| Num workers | 2 |
| Scheduler | CosineAnnealingLR per stage (stepped once per epoch) |
| Sampler | WeightedRandomSampler (inverse-frequency, power=1.0) |
| Weight decay | 1e-4 |

### Selection

| Item | Value |
| --- | --- |
| Selection | 0.55 * QWK + 0.30 * Macro-F1 + 0.15 * Macro-AP |
| Checkpoints | best_model.pth (max selection), last_model.pth (every epoch) |


# SE-ResNeXt-50 Original Published Crops — 3-Stage Training

Train SE-ResNeXt-50 on original published 224x224 crops.
3-stage: frozen(5) -> coarse-tune(15) -> fine-tune(10).
Matches the working original-crop baseline pattern.


## 0. Setup


In [1]:
!pip -q install 'timm>=1.0' 'h5py>=3.9'


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, cohen_kappa_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm


Mounted at /content/drive


## 1. Configuration


In [3]:
# ─── Paths ───────────────────────────────────────────────────────────────────
DATASET_ROOT = Path(
    '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/'
    'extracted/KneeXrayData/ClsKLData/kneeKL224'
)

# ─── Training ─────────────────────────────────────────────────────────────────
SEED = 42
INPUT_SIZE = 384
BATCH_SIZE = 48
NUM_WORKERS = 2
EPOCHS_S1, EPOCHS_S2, EPOCHS_S3 = 5, 15, 10
TOTAL_EPOCHS = EPOCHS_S1 + EPOCHS_S2 + EPOCHS_S3
WEIGHT_DECAY = 1e-4
LR_HEAD_S1 = 3e-4
LR_HEAD_S2 = 3e-4
LR_BACKBONE_S2 = 3e-5
LR_S3 = 1e-5
SAMPLER_POWER = 1.0

# ─── Derived ────────────────────────────────────────────────────────────────
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_%f_UTC')
RUN_DIR = Path('/content/drive/MyDrive/Models/seresnext50_32x4d_original') / RUN_TIMESTAMP

for p in (DATASET_ROOT,):
    if not p.exists():
        raise FileNotFoundError(p)
RUN_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print(f'Stages: {EPOCHS_S1}/{EPOCHS_S2}/{EPOCHS_S3} epochs  Total: {TOTAL_EPOCHS}')
print(f'LR: S1={LR_HEAD_S1}  S2={LR_HEAD_S2}/{LR_BACKBONE_S2}  S3={LR_S3}')
print(f'Batch size: {BATCH_SIZE}  Workers: {NUM_WORKERS}')
print(f'Scheduler: CosineAnnealingLR per stage')


Device: cuda
Stages: 5/15/10 epochs  Total: 30
LR: S1=0.0003  S2=0.0003/3e-05  S3=1e-05
Batch size: 48  Workers: 2
Scheduler: CosineAnnealingLR per stage


## 2. Load train / val splits


In [4]:
def load_split(root, split):
    paths, labels = [], []
    for grade in range(5):
        gdir = root / split / str(grade)
        if not gdir.exists():
            continue
        for img in sorted(gdir.glob('*.png')):
            paths.append(str(img))
            labels.append(grade)
    print(f'  Loaded {split}: {len(paths)} images')
    return paths, labels

print(f'Dataset: {DATASET_ROOT}')
train_paths, train_labels = load_split(DATASET_ROOT, 'train')
val_paths,   val_labels   = load_split(DATASET_ROOT, 'val')

class_counts = np.bincount(train_labels, minlength=5)
print(f'Class counts: {dict(enumerate(class_counts))}')


Dataset: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224
  Loaded train: 5778 images
  Loaded val: 826 images
Class counts: {0: np.int64(2286), 1: np.int64(1046), 2: np.int64(1516), 3: np.int64(757), 4: np.int64(173)}


## 3. Preprocessing & Dataset


In [5]:
class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

class SquarePad:
    def __call__(self, image_rgb):
        h, w = image_rgb.shape[:2]
        side = max(h, w)
        top = (side - h) // 2
        left = (side - w) // 2
        return cv2.copyMakeBorder(
            image_rgb, top, side - h - top, left, side - w - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0))

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.50),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.05), ratio=(0.5, 2.0), value=0),
    normalize,
])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    normalize,
])

class PublishedCropDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        img = cv2.imread(self.paths[index])
        if img is None:
            raise IOError(f'Cannot read: {self.paths[index]}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(img), int(self.labels[index])


class SEResNeXt50Model(nn.Module):
    def __init__(self):
        super().__init__()
        # NOTE: use classification mode (NOT features_only) so the original
        # layer1..layer4 attributes are preserved on the backbone — Stage 2
        # fine-tuning needs to unfreeze the last conv block by name.
        self.backbone = timm.create_model(
            'seresnext50_32x4d', pretrained=True, num_classes=0,
        )
        channels = self.backbone.num_features
        # Drop the timm classifier; we use our own nn.Linear on global-pooled features.
        self.classifier = nn.Linear(channels, 5)

    def forward(self, images):
        # timm's num_classes=0 model already does global pooling → (B, channels)
        features = self.backbone(images)
        return self.classifier(features)

    def freeze_all(self):
        for p in self.parameters():
            p.requires_grad = False
        for p in self.classifier.parameters():
            p.requires_grad = True
        print('  [Stage 1] Frozen backbone, training classifier head only')

    def unfreeze_last_block(self):
        # timm ResNet-family models expose layer1..layer4; the "last conv block"
        # is layer4. We unfreeze that plus the classifier.
        for p in self.parameters():
            p.requires_grad = False
        for p in self.backbone.layer4.parameters():
            p.requires_grad = True
        for p in self.classifier.parameters():
            p.requires_grad = True
        print('  [Stage 2] Unfrozen last conv block (layer4), training last block + classifier')

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad = True
        print('  [Stage 3] Full model unfrozen, fine-tuning end-to-end')


model = SEResNeXt50Model().to(DEVICE)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total:,}  Trainable: {trainable:,}')

weights = (1.0 / np.power(class_counts, SAMPLER_POWER))[train_labels]
sampler = WeightedRandomSampler(
    torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)

train_dataset = PublishedCropDataset(train_paths, train_labels, train_transform)
val_dataset   = PublishedCropDataset(val_paths,   val_labels,   val_transform)
VAL_BATCH = BATCH_SIZE * 2

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(
    val_dataset, batch_size=VAL_BATCH, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B /  111MB            

model.safetensors: downloading bytes:           |  0.00B            

Total parameters: 25,521,141  Trainable: 25,521,141
Train batches: 121  Val batches: 9


## 4. Training Loop


In [6]:
# ─── Option A: Ordinal soft-label CE + Mixup + TTA ──────────────────────
# Added after the ablation findings: ordinal soft-label +0.01 QWK, mixup is a
# strong minority-class regularizer, and TTA gives a free inference lift.
ORDINAL_SIGMA = 0.70   # sigma for Gaussian ordinal soft targets (matches the
                       # se_resnext50 ablation winner 'final_native_cam_ordinal_soft_label')
MIXUP_ALPHA = 0.40     # beta distribution alpha for mixup (mild, KL-KL adjacency)
USE_ORDINAL_LOSS = True   # set False to fall back to plain nn.CrossEntropyLoss
USE_MIXUP = True          # set False to skip mixup (criterion falls back to hard CE)
USE_TTA = True            # set False to use single-view evaluation


def ordinal_soft_targets(labels, num_classes=5, sigma=ORDINAL_SIGMA):
    """Build Gaussian ordinal soft targets around the integer grade.

    Grade k -> [exp(-d^2 / 2σ^2)] for d = 0..num_classes-1, then row-normalize.
    Pixels closer to the true grade get higher mass; far grades get near-zero.
    This encodes the KL ordering directly in the target distribution.
    """
    device = labels.device
    indices = torch.arange(num_classes, device=device, dtype=torch.float32)
    distance = (indices.unsqueeze(0) - labels.unsqueeze(1).float()).abs()
    weights = torch.exp(-(distance ** 2) / (2.0 * sigma * sigma))
    return weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-12)


def ordinal_soft_cross_entropy(logits, soft_targets):
    """Cross-entropy against continuous soft targets (row-wise KL)."""
    log_probs = F.log_softmax(logits, dim=1)
    return -(soft_targets * log_probs).sum(dim=1).mean()


def mixup_batch(images, labels, alpha=MIXUP_ALPHA):
    """Mixup: blend images and (soft) labels from a paired random permutation.

    Returns the mixed images and the *combined* target distribution (hard+soft blend)
    so the loss function can consume a single target tensor per example.
    Returns (images, combined_targets, lam) so the caller can decide between
    ordinal soft CE and plain CE (the latter needs integer targets from one side).
    """
    if alpha <= 0:
        raise ValueError("mixup alpha must be > 0")
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(images.size(0), device=images.device)
    mixed_images = lam * images + (1.0 - lam) * images[perm]
    return mixed_images, labels, labels[perm], lam


def compute_loss(logits, labels_a, labels_b, lam, use_ordinal, use_mixup):
    """Compute the training loss, handling ordinal vs plain and mixup vs clean."""
    if use_mixup:
        if use_ordinal:
            t_a = ordinal_soft_targets(labels_a)
            t_b = ordinal_soft_targets(labels_b)
            blended = lam * t_a + (1.0 - lam) * t_b
            return ordinal_soft_cross_entropy(logits, blended)
        # Plain CE on mixed inputs: average of two CE terms (standard mixup recipe)
        return lam * F.cross_entropy(logits, labels_a) + (1.0 - lam) * F.cross_entropy(logits, labels_b)
    # No mixup
    if use_ordinal:
        return ordinal_soft_cross_entropy(logits, ordinal_soft_targets(labels_a))
    return F.cross_entropy(logits, labels_a)


def evaluate_with_tta(loader, model, num_classes=5):
    """TTA inference: average softmax over original + horizontal flip.

    Returns the same 11-key metric dict as evaluate_full so the training loop
    can use metrics[...] keys interchangeably with or without TTA.
    """
    model.eval()
    all_labels, all_preds, all_probas = [], [], []
    total_loss, total_samples = 0.0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits_orig = model(images).float()
            logits_flip = model(torch.flip(images, dims=[3])).float()
            avg_logits = 0.5 * (logits_orig + logits_flip)
            probas = F.softmax(avg_logits, dim=1).cpu().numpy()
            preds = avg_logits.argmax(dim=1).cpu().numpy()
            if USE_ORDINAL_LOSS:
                loss = ordinal_soft_cross_entropy(avg_logits, ordinal_soft_targets(labels))
            else:
                loss = F.cross_entropy(avg_logits, labels)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probas.extend(probas)
            total_loss += loss.item() * len(labels)
            total_samples += len(labels)

    y_true = np.asarray(all_labels).astype(int)
    y_pred = np.asarray(all_preds).astype(int)
    y_proba = np.asarray(all_probas)
    y_onehot = np.eye(num_classes)[y_true]

    qwk = float(cohen_kappa_score(y_true, y_pred, weights='quadratic'))
    _, _, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = float(f1_macro)
    macro_ap = float(average_precision_score(y_onehot, y_proba, average='macro'))
    macro_auc = float(roc_auc_score(y_onehot, y_proba, average='macro'))
    mae = float(mean_absolute_error(y_true, y_pred))
    off1_acc = float(np.mean(np.abs(y_true - y_pred) <= 1))
    accuracy = float(np.mean(y_true == y_pred))
    selection = 0.55 * qwk + 0.30 * macro_f1 + 0.15 * macro_ap

    per_class_f1 = {}
    for grade in range(num_classes):
        mask_t = y_true == grade
        mask_p = y_pred == grade
        tp = float(np.sum(mask_t & mask_p))
        fp = float(np.sum((y_true != grade) & mask_p))
        fn = float(np.sum(mask_t & (y_pred != grade)))
        p_g = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r_g = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f_g = 2 * p_g * r_g / (p_g + r_g) if (p_g + r_g) > 0 else 0.0
        per_class_f1[grade] = {
            "precision": p_g, "recall": r_g, "f1": f_g,
            "support": int(mask_t.sum()),
        }

    return {
        "loss": total_loss / total_samples,
        "accuracy": accuracy,
        "qwk": qwk,
        "mae": mae,
        "off1_acc": off1_acc,
        "macro_f1": macro_f1,
        "macro_ap": macro_ap,
        "macro_auc": macro_auc,
        "selection": float(selection),
        "per_class_f1": per_class_f1,
        "probas": y_proba,
    }

Option A active: ordinal_sigma=0.7, mixup_alpha=0.4, ordinal=True, mixup=True, tta=True


In [7]:
def make_optimizer(lr, backbone_lr=None):
    head = list(model.classifier.parameters())
    bb   = [p for n, p in model.named_parameters()
            if 'classifier' not in n and p.requires_grad]
    if backbone_lr is None:
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    return torch.optim.AdamW([
        {'params': head, 'lr': lr},
        {'params': bb,   'lr': backbone_lr},
    ], weight_decay=WEIGHT_DECAY)

def make_scheduler(optimizer, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7)

def evaluate(loader):
    labels, probabilities = [], []
    model.eval()
    with torch.inference_mode():
        for images, batch_labels in tqdm(loader, desc='Evaluating'):
            probs = F.softmax(
                model(images.to(DEVICE, non_blocking=True)).float(), dim=1
            ).cpu().numpy()
            labels.extend(batch_labels.numpy())
            probabilities.extend(probs)
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    predictions = probabilities.argmax(axis=1)
    _, _, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0)
    ap  = average_precision_score(np.eye(5)[labels], probabilities, average='macro')
    qwk = cohen_kappa_score(labels, predictions, weights='quadratic')
    return {'qwk': float(qwk), 'f1': float(f1), 'ap': float(ap),
            'selection': float(0.55 * qwk + 0.30 * f1 + 0.15 * ap)}

best_selection = -float('inf')
history = []
scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')

for stage_idx, (stage_name, stage_epochs, head_lr, bb_lr) in enumerate([
    (f'Stage 1 -- Head Only ({EPOCHS_S1} epochs)', EPOCHS_S1, LR_HEAD_S1, None),
    (f'Stage 2 -- Coarse Tune ({EPOCHS_S2} epochs)', EPOCHS_S2, LR_HEAD_S2, LR_BACKBONE_S2),
    (f'Stage 3 -- Fine-Tune ({EPOCHS_S3} epochs)', EPOCHS_S3, LR_S3, LR_S3),
], start=1):
    print(f'\n{"=" * 60}')
    print(f' {stage_name}')
    print(f'{"=" * 60}')

    if stage_idx == 1:
        model.freeze_all()
    elif stage_idx == 2:
        model.unfreeze_last_block()
    else:
        model.unfreeze_all()

    optimizer = make_optimizer(head_lr, bb_lr)
    scheduler = make_scheduler(optimizer, stage_epochs)

    for epoch in range(stage_epochs):
        model.train()
        loss_sum, samples = 0.0, 0
        for images, labels in tqdm(train_loader, desc=f'S{stage_idx}E{epoch+1}'):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                if USE_MIXUP:
                    images_m, ya, yb, lam = mixup_batch(images, labels)
                    logits = model(images_m)
                    loss = compute_loss(logits, ya, yb, lam, USE_ORDINAL_LOSS, True)
                else:
                    logits = model(images)
                    loss = compute_loss(logits, labels, labels, 1.0, USE_ORDINAL_LOSS, False)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * len(labels)
            samples += len(labels)

        scheduler.step()  # one step per epoch

        metrics = evaluate_with_tta(val_loader, model) if USE_TTA else evaluate(val_loader)
        train_loss = loss_sum / samples
        row = {
            'stage': stage_idx, 'epoch': epoch + 1,
            'train_loss': train_loss, 'val_loss': metrics['selection'],
            'qwk': metrics['qwk'], 'f1': metrics['f1'], 'ap': metrics['ap'],
            'selection': metrics['selection'],
            'lr_head': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        print(json.dumps(row, indent=2))

        payload = {
            'model_state_dict': model.state_dict(),
            'architecture': 'seresnext50_32x4d_original_option_a',
            'loss_type': 'ordinal_soft_ce_mixup_tta',
            'stage': stage_idx, 'epoch': epoch + 1,
            'selection': metrics['selection'],
            'qwk': metrics['qwk'], 'f1': metrics['f1'], 'ap': metrics['ap'],
            'history': history,
        }

        torch.save(payload, RUN_DIR / 'last_model.pth')

        if metrics['selection'] > best_selection:
            best_selection = metrics['selection']
            torch.save(payload, RUN_DIR / 'best_model.pth')
            print(f'  -> New best! selection={best_selection:.4f} QWK={metrics["qwk"]:.4f}')

print(f'\nBest selection: {best_selection:.4f}')
print(f'Best:   {RUN_DIR / "best_model.pth"}')
print(f'Last:   {RUN_DIR / "last_model.pth"}')



 Stage 1 -- Head Only (5 epochs)
  [Stage 1] Frozen backbone, training classifier head only


S1E1:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "stage": 1,
  "epoch": 1,
  "train_loss": 1.5970052577253442,
  "val_loss": 0.17155960803190562,
  "qwk": 0.1616183238788398,
  "f1": 0.11813652650575521,
  "ap": 0.31485714631211453,
  "selection": 0.17155960803190562,
  "lr_head": 0.00027136209830652333
}
  -> New best! selection=0.1716 QWK=0.1616


S1E2:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "stage": 1,
  "epoch": 2,
  "train_loss": 1.5695784287536874,
  "val_loss": 0.27143793195938004,
  "qwk": 0.28324882127924567,
  "f1": 0.20812398165012183,
  "ap": 0.35475923840505585,
  "selection": 0.27143793195938004,
  "lr_head": 0.00019638709830652334
}
  -> New best! selection=0.2714 QWK=0.2832


...

{
  "stage": 2,
  "epoch": 9,
  "train_loss": 1.302902281098648,
  "val_loss": 0.6781210547097075,
  "qwk": 0.7436710685658267,
  "f1": 0.5887889416026972,
  "ap": 0.6164352301179579,
  "selection": 0.6781210547097075,
  "lr_head": 0.00010371290169347662
}
  -> New best! selection=0.6781 QWK=0.7437


S2E10:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "stage": 2,
  "epoch": 10,
  "train_loss": 1.3100549698123556,
  "val_loss": 0.6776184326322636,
  "qwk": 0.742307924802031,
  "f1": 0.5873753202447107,
  "ap": 0.6209098527848886,
  "selection": 0.6776184326322636,
  "lr_head": 7.507500000000001e-05
}


S2E11:   0%|          | 0/121 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 5. Save History & Metadata


In [ ]:
pd.DataFrame(history).to_csv(RUN_DIR / 'history.csv', index=False)

metadata = {
    'architecture': 'seresnext50_32x4d_original',
    'loss': 'cross_entropy',
    'stages': [EPOCHS_S1, EPOCHS_S2, EPOCHS_S3],
    'lr_head': [LR_HEAD_S1, LR_HEAD_S2, LR_S3],
    'lr_backbone': [0, LR_BACKBONE_S2, LR_S3],
    'weight_decay': WEIGHT_DECAY,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'input_size': INPUT_SIZE,
    'scheduler': 'cosine_annealing',
    'sampler_power': SAMPLER_POWER,
    'best_selection': best_selection,
    'dataset_root': str(DATASET_ROOT),
    'train_samples': len(train_paths),
    'val_samples': len(val_paths),
}
with open(RUN_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('Saved:')
for fn in ['best_model.pth', 'last_model.pth', 'history.csv', 'metadata.json']:
    print(f'  {RUN_DIR / fn}')
print(f'\nBest selection: {best_selection:.4f}')
